In [ ]:
# Cell 1 - Install
!pip install ultralytics supervision roboflow opencv-python-headless -q

In [ ]:
# Cell 2 - Download dataset from Roboflow
from roboflow import Roboflow

rf = Roboflow(api_key="p2WbAXsTeICMQ1ub7nSX")
project = rf.workspace("raqib-jokhio").project("helmet-detection-nsbwm-ttpaj")
version = project.version(1)
dataset = version.download("yolov8")
print("Dataset location:", dataset.location)

In [ ]:
# Cell 3 - Fix data.yaml (no valid folder in dataset)
import yaml

data = {
    'names': ['Helmet', 'Motorbike', 'NoHelmet', 'PNumber'],
    'nc': 4,
    'train': '/content/Helmet-detection-1/train/images',
    'val': '/content/Helmet-detection-1/train/images',
}

with open('/content/Helmet-detection-1/data.yaml', 'w') as f:
    yaml.dump(data, f)

print('data.yaml fixed')

In [ ]:
# Cell 4 - Train
!yolo task=detect mode=train \
    model=yolov8n.pt \
    data=/content/Helmet-detection-1/data.yaml \
    epochs=20 \
    imgsz=640 \
    batch=16 \
    name=helmet_model \
    project=/content/runs \
    exist_ok=True
print('Training done!')

In [ ]:
# Cell 5 - Verify model saved
import os
model_path = '/content/runs/helmet_model/weights/best.pt'
if os.path.exists(model_path):
    print('Model found:', model_path)
else:
    print('Model NOT found - training may have failed')
    for root, dirs, files in os.walk('/content/runs'):
        for f in files:
            print(os.path.join(root, f))

In [ ]:
# Cell 6 - Upload video
from google.colab import files
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f'Uploaded: {video_path}')

In [ ]:
# Cell 7 - Run helmet detection
import cv2
import supervision as sv
from ultralytics import YOLO
import os
import zipfile

model = YOLO('/content/runs/helmet_model/weights/best.pt')
CLASS_NAMES = model.names
print('Classes:', CLASS_NAMES)

NO_HELMET_IDS = [k for k, v in CLASS_NAMES.items() if 'no' in v.lower()]
print('No-helmet class IDs:', NO_HELMET_IDS)

output_path = 'output_helmet.mp4'
violations_dir = 'helmet_violations'
os.makedirs(violations_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

violation_log = []
logged_frames = set()
frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)

    for i, cls_id in enumerate(detections.class_id):
        if int(cls_id) in NO_HELMET_IDS and frame_count not in logged_frames:
            logged_frames.add(frame_count)
            snapshot_path = f'{violations_dir}/helmet_violation_{frame_count}.jpg'
            cv2.imwrite(snapshot_path, frame)
            violation_log.append({'frame': frame_count, 'snapshot': snapshot_path})
            print(f'HELMET VIOLATION at frame {frame_count}')

    labels = [
        f"{CLASS_NAMES[int(cls)]} {conf:.2f}"
        for cls, conf in zip(detections.class_id, detections.confidence)
    ]

    frame = box_annotator.annotate(scene=frame, detections=detections)
    frame = label_annotator.annotate(scene=frame, detections=detections, labels=labels)

    out.write(frame)
    frame_count += 1
    if frame_count % 50 == 0:
        print(f'Processed {frame_count} frames...')

cap.release()
out.release()
print(f'Done. Total helmet violations: {len(violation_log)}')

In [8]:
# Cell 8 - Download output
from google.colab import files
import zipfile, os

if os.listdir('helmet_violations'):
    with zipfile.ZipFile('helmet_violations.zip', 'w') as zf:
        for f in os.listdir('helmet_violations'):
            zf.write(os.path.join('helmet_violations', f), f)
    files.download('helmet_violations.zip')

files.download('output_helmet.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>